# S2 · Muestreo, cuantización y registro · ¿Cada cuánto registrar sin que el registro mienta?
**Sistemas de adquisición de datos · TSU en Electrónica y Pruebas · Jabil**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggt13/DAQ_TSU/blob/main/sesion_02_muestreo_y_cuantizacion/S2_AC_CLASE_Colab_ALUMNO.ipynb)

## Objetivo
Elegir el **intervalo de registro** y estimar el **error de cuantización** de un registrador para un proceso de planta,
y comprobar con datos reales cómo un intervalo mal elegido inventa un ciclo que no existe (**aliasing**).

## La situación
Vas a dejar un registrador Elitech RC-5 siete días en un lugar de la planta. Cada alumno tiene un escenario distinto
(almacén, refrigerador, cuarto de servidores…), con su propio fenómeno más rápido y su periodo. Antes de programar el
registrador hay que contestar: ¿cada cuánto como máximo?, ¿qué pasa si me equivoco?, ¿qué tan fino mide? y ¿alcanza
la memoria?

## En qué consiste este cuaderno
| Parte | Tiempo | Qué haces | Qué produce |
|---|---|---|---|
| **A · Actividad en clase** | 30 min | Cuatro ejercicios (★ primero) sobre **tu** escenario. En cada uno escribes tu resultado, la sustitución con tus números y su unidad, y lo que predice. Al final eliges el intervalo de tu campaña | Tus resultados, tus predicciones y el intervalo que programarás en el RC-5 |
| **B · Práctica** | 80 min | Tomas 60 días de temperatura real de Guadalajara, la registras con un intervalo demasiado largo y mides el ciclo falso que aparece | Gráficas firmadas, tu conclusión y el archivo `S2_<código>.csv` |
| **Hoja de resultados** | en la práctica | Importas el CSV en tu hoja de cálculo de Google y contestas las preguntas | El reporte que entregas |

## Qué entregas y cómo se califica
1. **Este cuaderno**, al terminar la sesión, con todo ejecutado. ★ 3 pts · ejercicios 2, 3 y 4, 2 pts cada uno ·
   cierre (intervalo de la campaña) 1 pt. Se califica que el resultado sea correcto para **tus** datos y que la
   sustitución y la predicción sean tuyas.
2. **La hoja de resultados** con el reporte, en 7 días naturales.
3. **Tu ficha de campaña** (al final del cuaderno), que sirve para elegir las tres campañas del grupo.

## Reglas
- **Archivo → Guardar una copia en Drive** antes de empezar; trabaja en tu copia.
- Completa sólo lo marcado con ✎. Ninguna celda avanza si falta algo: el mensaje dice qué.
- La revisión automática dice si el ejercicio está **completo**, no si está **bien**; eso lo califica el profesor.
- Si usas un asistente de IA, dilo en tu reporte. Las sustituciones y las respuestas llevan **tus** números.

## Cómo se usa este cuaderno · no necesitas saber programar
Sólo cambias lo que está después del signo **=** en los renglones marcados con ✎.

| Para… | Haz esto |
|---|---|
| Ejecutar una celda | Clic en ▶ a su izquierda (o Ctrl + Enter). Ejecuta las celdas en orden, de arriba hacia abajo |
| Escribir un número | Después del **=**, con punto decimal y sin comas de miles: `u_A = 0.0316`, `MEMORIA = 32000`. Usa al menos 3 cifras significativas |
| Escribir un texto | Entre comillas: `SUST_1 = "70 / 2^10 = 0.0684 °C"`. No borres las comillas |
| Calcular | Con calculadora o con tu hoja de cálculo. Si sabes Python puedes escribir la operación en vez del número: `+ − * /`, potencia `**`, raíz `np.sqrt(x)`, eˣ `np.exp(x)` |
| Ver tus datos | Están impresos debajo de la celda «Tus datos»; usa esos números |
| Un mensaje en rojo | Lee la **última línea**: dice qué falta. Corrige y vuelve a ejecutar esa celda |
| Antes de entregar | **Entorno de ejecución → Reiniciar y ejecutar todo**, y revisa que todo salga ✔ |

In [ ]:
#@title Herramientas del curso (no modificar)
"""Herramientas comunes del curso «Sistemas de adquisición de datos» 2027.

Identificación del alumno, datos personales a partir de su código, firma de las gráficas,
huella de los resultados y CSV firmado para la hoja de resultados.

Este archivo es la fuente única: los cuadernos de cada sesión copian su contenido en la celda
«Herramientas del curso». Si se corrige aquí, se regeneran los cuadernos.
"""
import csv
import datetime as _dt
import hashlib
import zlib

import numpy as np

try:
    from zoneinfo import ZoneInfo
    _ZONA = ZoneInfo("America/Mexico_City")
except Exception:          # Python sin zoneinfo: se usa la hora del sistema
    _ZONA = None

_VALORES_DE_EJEMPLO = {"", "TU NOMBRE", "NOMBRE APELLIDO", "000000000", "123456789"}


class Alumno:
    """Datos de quien ejecuta el cuaderno. Se crea con identificar()."""

    def __init__(self, nombre, codigo, sesion):
        self.nombre = nombre
        self.codigo = codigo
        self.sesion = sesion

    @property
    def semilla(self):
        return int(self.codigo) % (2 ** 32)

    def rng(self, etiqueta=""):
        """Generador de números aleatorios propio del alumno.

        Cada etiqueta da una secuencia independiente: agregar un parámetro nuevo
        no cambia los que ya existían.
        """
        return np.random.default_rng([self.semilla, zlib.crc32(etiqueta.encode("utf-8"))])

    def __repr__(self):
        return f"Alumno({self.nombre!r}, {self.codigo!r}, {self.sesion!r})"


def identificar(nombre, codigo, sesion):
    """Valida la celda de identificación y devuelve el Alumno.

    Detiene el cuaderno con un mensaje claro si el nombre o el código faltan
    o se dejaron con el valor de ejemplo.
    """
    nombre = " ".join(str(nombre).split())
    codigo = str(codigo).strip()
    if nombre.upper() in _VALORES_DE_EJEMPLO or len(nombre.split()) < 2:
        raise ValueError("Escribe tu nombre completo (nombre y apellido) en NOMBRE y vuelve a ejecutar.")
    if codigo in _VALORES_DE_EJEMPLO or not codigo.isdigit() or not 7 <= len(codigo) <= 10:
        raise ValueError("Escribe tu código de alumno (sólo dígitos, de 7 a 10) en CODIGO y vuelve a ejecutar.")
    return Alumno(nombre, codigo, sesion)


def parametro(alumno, etiqueta, minimo, maximo, decimales=3):
    """Un valor personal entre minimo y maximo, siempre el mismo para el mismo código."""
    valor = alumno.rng(etiqueta).uniform(minimo, maximo)
    return round(float(valor), decimales)


def asignar(alumno, opciones, salto=0):
    """Elige una opción de la lista según el código (estación, ventana de fechas, carga del motor...)."""
    opciones = list(opciones)
    return opciones[(int(alumno.codigo) + salto) % len(opciones)]


def _redondea(valor):
    if isinstance(valor, (int, float, np.floating, np.integer)):
        return format(float(valor), ".4g")
    return str(valor)


def huella(alumno, resultados):
    """Seis caracteres que dependen del código, la sesión y los resultados (redondeados a 4 cifras)."""
    partes = [alumno.codigo, alumno.sesion] + [f"{k}={_redondea(v)}" for k, v in sorted(resultados.items())]
    return hashlib.sha256("|".join(partes).encode("utf-8")).hexdigest()[:6].upper()


def ahora():
    momento = _dt.datetime.now(_ZONA) if _ZONA else _dt.datetime.now()
    return momento.strftime("%Y-%m-%d %H:%M")


def titulo(ax, texto, alumno):
    """Título de la gráfica con el nombre del alumno."""
    ax.set_title(f"{texto} · {alumno.nombre}")


def firma(fig, alumno, resultados=None):
    """Pie de la figura: nombre · código · sesión · fecha y hora · huella.

    Toda gráfica que se entrega debe pasar por aquí.
    """
    texto = f"{alumno.nombre} · {alumno.codigo} · {alumno.sesion} · {ahora()}"
    if resultados:
        texto += f" · huella {huella(alumno, resultados)}"
    fig.text(0.01, 0.005, texto, fontsize=8, color="0.35", ha="left", va="bottom")
    fig.subplots_adjust(bottom=max(fig.subplotpars.bottom, 0.14))
    fig._firmada = True
    return fig


def exportar_csv(ruta, alumno, resultados, columnas, extra=None):
    """Escribe el CSV firmado que se importa en la pestaña «CSV» de la hoja de resultados.

    Renglones de metadatos: nombre, codigo, sesion, huella, ejecutado y los de `extra` (p. ej. la fuente
    de los datos). Después, un renglón por resultado («resultado:<nombre>», valor). Luego un renglón vacío
    y la tabla: encabezados y datos. `columnas` es un diccionario {encabezado: lista de valores}, todas de
    la misma longitud. La hoja busca cada dato por su etiqueta, no por su renglón.
    """
    claves = [k.lower() for k in resultados]
    if len(set(claves)) != len(claves):    # la hoja busca sin distinguir mayúsculas: U y u serían el mismo dato
        raise ValueError("Dos resultados tienen el mismo nombre salvo mayúsculas; renómbrelos.")
    largos = {len(v) for v in columnas.values()}
    if len(largos) != 1:
        raise ValueError("Todas las columnas de la tabla deben tener el mismo número de renglones.")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["nombre", alumno.nombre])
        w.writerow(["codigo", alumno.codigo])
        w.writerow(["sesion", alumno.sesion])
        w.writerow(["huella", huella(alumno, resultados)])
        w.writerow(["ejecutado", ahora()])
        for k, v in (extra or {}).items():
            w.writerow([k, v])
        for k, v in resultados.items():
            w.writerow([f"resultado:{k}", _redondea(v)])
        w.writerow([])
        w.writerow(list(columnas))
        for fila in zip(*columnas.values()):
            w.writerow([_redondea(x) for x in fila])
    return ruta


def leer_csv_firmado(ruta):
    """Para el profesor: devuelve (metadatos, resultados, tabla) de un CSV firmado."""
    meta, resultados, tabla = {}, {}, []
    with open(ruta, newline="", encoding="utf-8") as f:
        filas = list(csv.reader(f))
    i = 0
    while i < len(filas) and filas[i]:
        clave, valor = filas[i][0], filas[i][1]
        if clave.startswith("resultado:"):
            resultados[clave.split(":", 1)[1]] = float(valor)
        else:
            meta[clave] = valor
        i += 1
    tabla = filas[i + 1:] if i + 1 < len(filas) else []
    return meta, resultados, tabla


def verificar_csv(ruta):
    """Para el profesor: True si la huella del CSV corresponde a su código, sesión y resultados."""
    meta, resultados, _ = leer_csv_firmado(ruta)
    alumno = Alumno(meta["nombre"], meta["codigo"], meta["sesion"])
    return huella(alumno, resultados) == meta["huella"]


# ---------------------------------------------------------------------------------------------
# Revisión de lo que escribe el alumno. Comprueba que cada ejercicio esté COMPLETO, no que esté
# bien: el resultado correcto lo califica el profesor con la clave. Si falta algo, detiene el
# cuaderno y dice qué falta.
# ---------------------------------------------------------------------------------------------
import math
import re

REGISTRO = {}


def _normaliza(texto):
    t = str(texto).replace("\u202f", "").replace("\u00a0", "").replace(" ", "")
    t = re.sub(r"(?<![\d.])([1-9]\d{0,2})((?:,\d{3})+)(?![\d])",  # separador de miles: 32,000 -> 32000
               lambda m: m.group(1) + m.group(2).replace(",", ""), t)
    return t.replace(",", ".")                                       # coma decimal: 0,068 -> 0.068


def _formas(x):
    """Maneras razonables de escribir un número: entero, o redondeado a 2, 3 o 4 cifras significativas."""
    x = float(x)
    formas = set()
    if x == int(x) and abs(x) < 1e9:
        formas.add(str(int(x)))
    if abs(x) >= 100:
        formas.add(str(int(round(x))))                       # 34271.6 escrito como 34272
    if abs(x) >= 1:
        for dec in (1, 2):
            formas.add(f"{x:.{dec}f}".rstrip("0").rstrip("."))  # 123456.7 con sus decimales
    for cifras in (2, 3, 4, 5, 6):
        g = f"{x:.{cifras}g}"
        if "e" not in g:
            formas.add(g)
    return {f for f in formas if f not in ("", "-0")}


def aparece(x, texto):
    """True si el número x está escrito en el texto (sin confundir 0.03 con 0.0337)."""
    t = _normaliza(texto)
    def patron(f):
        cola = r"0*" if "." in f else r"(?:\.0*)?"            # 0.0342 = 0.034200 · 120 = 120.0, pero no 1200
        return r"(?<![\d.])" + re.escape(f) + cola + r"(?![\d])"
    return any(re.search(patron(f), t) for f in _formas(x))


def _palabras(texto):
    return len(re.findall(r"\w+", str(texto)))


def revisa(n, valor, sustitucion, prediccion, datos, unidades):
    """Revisión de un ejercicio de la parte A.

    valor: lo que calculó la expresión del alumno. sustitucion: texto con sus números, el resultado y la
    unidad (p. ej. «70 / 2**8 = 0.273 °C»). prediccion: qué espera ver en la práctica, con sus palabras.
    datos: los números personales que deben aparecer en la sustitución. unidades: unidades aceptadas.
    """
    faltan = []
    if valor is None:
        faltan.append("la expresión en Python")
    elif not isinstance(valor, (int, float, np.integer, np.floating)) or not math.isfinite(float(valor)):
        faltan.append("una expresión que dé un número")
    s = str(sustitucion)
    if "=" not in s:
        faltan.append("la sustitución con «=»")
    # datos: lista de números (se nombran por su valor) o {descripción: valor} (se nombran sin revelar el valor)
    pares = datos.items() if isinstance(datos, dict) else [(None, d) for d in datos]
    for que, d in pares:
        if not aparece(d, s):
            faltan.append(f"{que} en la sustitución" if que else f"tu dato {float(d):g} en la sustitución")
    if valor is not None and not faltan and not aparece(valor, s.split("=")[-1]):
        faltan.append("el resultado que dio tu expresión, después del último «=»")
    if not any(u in s for u in unidades):
        faltan.append("la unidad (" + " o ".join(unidades) + ")")
    if _palabras(prediccion) < 10:
        faltan.append("lo que predice, en al menos 10 palabras")
    REGISTRO[n] = not faltan
    if faltan:
        raise ValueError(f"Ejercicio {n}: falta " + "; ".join(faltan) + ".")
    print(f"✔ Ejercicio {n} completo.")


def revisa_texto(nombre, texto, numeros, minimo=20):
    """Revisión de una respuesta escrita: largo mínimo y los números propios que debe citar.

    numeros: diccionario {qué es: valor}; el mensaje nombra lo que falta sin revelar el valor.
    """
    faltan = [que for que, x in numeros.items() if not aparece(x, texto)]
    if _palabras(texto) < minimo:
        faltan.insert(0, f"al menos {minimo} palabras")
    REGISTRO[nombre] = not faltan
    if faltan:
        raise ValueError(f"{nombre}: falta " + "; ".join(faltan) + ".")
    print(f"✔ {nombre} completo.")


import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 3.6), 'axes.spines.top': False, 'axes.spines.right': False})

def completa(valor, que):
    if valor is None:
        raise ValueError(f'Completa {que} (✎) y vuelve a ejecutar.')
    return valor

MEMORIA = 32000              # lecturas del Elitech RC-5
DT_MIN_RC5_S = 10            # intervalo mínimo del RC-5, en segundos
CAMPANA_DIAS = 7
ESCENARIOS = [  # lugar, variable, fenómeno más rápido, periodo base (s)
    ("Almacén de componentes con aire acondicionado", "T", "ciclo del aire acondicionado", 900),
    ("Refrigerador de pasta de soldadura", "T", "ciclo del compresor", 1200),
    ("Cuarto de servidores", "T", "ciclo del aire acondicionado", 480),
    ("Línea SMT, ambiente", "T", "cambio de turno", 28800),
    ("Oficina con ventana al poniente", "T", "ciclo día–noche", 86400),
    ("Bodega de producto terminado", "T", "ciclo día–noche", 86400),
    ("Área de empaque junto a la puerta de andén", "T", "apertura de la puerta", 300),
    ("Cámara de prueba térmica con control on-off", "T", "ciclo del control", 90),
    ("Gabinete de control con ventilador por termostato", "T", "encendido del ventilador", 100),
    ("Almacén de reactivos con extractor temporizado", "T", "ciclo del extractor", 1800),
]
RANGOS = {"T": (-30.0, 70.0, "°C")}   # el RC-5 sólo mide temperatura


def datos_s2(alumno):
    """Los datos personales de la sesión 2."""
    lugar, variable, fenomeno, p_base = asignar(alumno, ESCENARIOS)
    d = dict(LUGAR=lugar, VARIABLE=variable, FENOMENO=fenomeno)
    d["P_S"] = int(round(p_base * parametro(alumno, "factor_P", 0.8, 1.25, 3)))
    d["DT_MAL_S"] = int(round(d["P_S"] * parametro(alumno, "mal", 0.55, 0.9, 3)))
    lo, hi, unidad = RANGOS[variable]
    d.update(R_MIN=lo, R_MAX=hi, RANGO=hi - lo, UNIDAD=unidad)
    d["N_BITS"] = asignar(alumno, [8, 10, 12], salto=2)
    d["DT_B_H"] = asignar(alumno, [20, 21, 27, 28, 30, 32], salto=3)
    return d


def clave_s2(alumno):
    """Resultados esperados de la parte A y de la predicción de la parte B."""
    d = datos_s2(alumno)
    f, fs = 1 / d["P_S"], 1 / d["DT_MAL_S"]
    fb, fsb = 1 / 24, 1 / d["DT_B_H"]
    return {
        "dt_max_s": d["P_S"] / 2,
        "T_aparente_s": 1 / abs(f - round(f / fs) * fs),
        "error_cuant": d["RANGO"] / 2 ** d["N_BITS"] / 2,
        "dias_memoria": MEMORIA * (d["P_S"] / 10) / 86400,
        "T_aparente_pred_h": 1 / abs(fb - round(fb / fsb) * fsb),
    }


def revisa_intervalo(dt, P_S):
    """Revisión del intervalo que el alumno elige para su campaña: cumple las tres reglas o dice cuál falla."""
    if dt is None:
        raise ValueError("Escribe el intervalo que programarías, en segundos.")
    faltan = []
    if dt < DT_MIN_RC5_S:
        faltan.append(f"el RC-5 no registra más rápido que cada {DT_MIN_RC5_S} s")
    if dt > P_S / 2:
        faltan.append("con ese intervalo hay aliasing: revisa el ejercicio 1")
    if MEMORIA * dt < CAMPANA_DIAS * 86400:
        faltan.append(f"la memoria se llena antes de {CAMPANA_DIAS} días: revisa el ejercicio 4")
    REGISTRO["Intervalo de la campaña"] = not faltan
    if faltan:
        raise ValueError("Intervalo de la campaña: " + "; ".join(faltan) + ".")
    print(f"✔ Intervalo de la campaña: {dt:g} s cumple las tres reglas.")


def periodo_dominante(t_h, y, minimo_h, maximo_h):
    """Periodo de la componente más fuerte de una serie muestreada a intervalo fijo (FFT con relleno de ceros)."""
    y = np.asarray(y, float)
    y = y - np.polyval(np.polyfit(t_h, y, 1), t_h)
    dt = t_h[1] - t_h[0]
    n = 16 * len(y)
    F = np.abs(np.fft.rfft(y, n))
    fr = np.fft.rfftfreq(n, dt)
    ok = (fr > 1 / maximo_h) & (fr < 1 / minimo_h)
    return 1 / fr[ok][np.argmax(F[ok])]

print('Herramientas del curso listas.')

## Identificación
Tu nombre y tu código aparecen en cada gráfica y en el CSV. Sin ellos el cuaderno no avanza.

In [ ]:
NOMBRE = ""   # ✎ tu nombre completo
CODIGO = ""   # ✎ tu código de alumno
SESION = "S2"

alumno = identificar(NOMBRE, CODIGO, SESION)
print("Cuaderno de", alumno.nombre, "· código", alumno.codigo)

## Parte A · Actividad en clase (30 min)
Ejecuta la celda para ver **tu** escenario.

**Cómo se llena cada ejercicio.** Ejemplo con otros datos (no son los tuyos):
```
dt_max_s = 300
SUST_1   = "600 s / 2 = 300 s"
PRED_1   = "si registro más espaciado que cinco minutos el ciclo del horno aparecerá falso en la gráfica"
```

In [ ]:
globals().update(datos_s2(alumno))
print(f"Tu escenario · {LUGAR}")
print(f"  Variable: {VARIABLE} · fenómeno más rápido: {FENOMENO} · periodo P_S = {P_S} s ({P_S / 60:.1f} min)")
print(f"  Registrador: rango de {R_MIN:g} a {R_MAX:g} {UNIDAD} (RANGO = {RANGO:g} {UNIDAD}) · convertidor de N_BITS = {N_BITS} bits")
print(f"  Memoria: MEMORIA = {MEMORIA} lecturas · intervalo mal elegido para el ejercicio 2: DT_MAL_S = {DT_MAL_S} s")

### ★ Ejercicio 1 · ¿Cada cuánto como máximo? (3 pts)
**Para qué sirve:** es la regla que separa un registro útil de uno que miente. Si el registrador guarda más espaciado
que esto, el ciclo del proceso desaparece o aparece falso.

Criterio de Nyquist: **al menos dos lecturas por periodo**, es decir, **Δt_max = P / 2**. Calcula `dt_max_s` en segundos.

In [ ]:
# ✎ 1a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
dt_max_s = None   # ✎ escribe aquí
# ✎ 1b · La sustitución con TUS números, el resultado y la unidad
SUST_1 = ""   # ✎ escribe aquí
# ✎ 1c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_1 = ""   # ✎ escribe aquí

revisa(1, dt_max_s, SUST_1, PRED_1, datos={'tu dato P_S': P_S}, unidades=['s'])

### Ejercicio 2 · ¿Qué ciclo falso muestra un registro mal hecho? (2 pts)
**Para qué sirve:** reconocer el aliasing en un registro real: un ciclo lento que no existe y que lleva a decisiones
de mantenimiento equivocadas.

Si registras cada `DT_MAL_S` segundos (más que P/2), la frecuencia que aparece es la **diferencia** entre la frecuencia
real (1/P) y la del registro (1/Δt). El periodo aparente es el inverso de esa diferencia, sin signo (valor
absoluto). Resultado en segundos.

In [ ]:
# ✎ 2a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
T_aparente_s = None   # ✎ escribe aquí
# ✎ 2b · La sustitución con TUS números, el resultado y la unidad
SUST_2 = ""   # ✎ escribe aquí
# ✎ 2c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_2 = ""   # ✎ escribe aquí

revisa(2, T_aparente_s, SUST_2, PRED_2, datos={'tu dato P_S': P_S, 'tu dato DT_MAL_S': DT_MAL_S}, unidades=['s'])

La gráfica muestra tu fenómeno (azul), las lecturas del registro mal hecho (puntos) y el ciclo que esas lecturas dibujan (línea discontinua): ese ciclo no existe. Si tu resultado del ejercicio 2 es correcto, la línea discontinua pasa por todos los puntos.

In [ ]:
t = np.linspace(0, 3 * T_aparente_s, 3000)
ts = np.arange(0, 3 * T_aparente_s, DT_MAL_S)
fig, ax = plt.subplots()
ax.plot(t / 60, np.sin(2 * np.pi * t / P_S), color="#9DB7D5", lw=1, label=f"{FENOMENO}, P = {P_S} s")
ax.plot(t / 60, -np.sin(2 * np.pi * t / T_aparente_s), "--", color="#D9822B", lw=1.5,
        label=f"ciclo falso que ve el registro, {T_aparente_s:.0f} s")
ax.plot(ts / 60, np.sin(2 * np.pi * ts / P_S), "o", color="#D9822B", ms=6, label=f"lecturas cada {DT_MAL_S} s")
ax.set_xlabel("tiempo (min)"); ax.set_yticks([]); ax.set_ylim(-1.15, 1.9)
ax.legend(loc="upper center", ncol=3, fontsize=8, frameon=False)
titulo(ax, "Un registro que miente", alumno)
firma(fig, alumno); plt.show()

### Ejercicio 3 · ¿Qué tan fino mide? (2 pts)
**Para qué sirve:** es el error que agrega el registrador aunque el sensor fuera perfecto.

El convertidor divide el `RANGO` en 2ᴺ escalones y cada lectura se redondea al escalón más cercano. Calcula el
**error máximo de cuantización** (lo más lejos que puede quedar una lectura del valor real), en las unidades de tu
variable (`UNIDAD`).

In [ ]:
# ✎ 3a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
error_cuant = None   # ✎ escribe aquí
# ✎ 3b · La sustitución con TUS números, el resultado y la unidad
SUST_3 = ""   # ✎ escribe aquí
# ✎ 3c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_3 = ""   # ✎ escribe aquí

revisa(3, error_cuant, SUST_3, PRED_3, datos={'tu dato RANGO': RANGO, 'tu dato N_BITS': N_BITS}, unidades=[UNIDAD])

### Ejercicio 4 · ¿Alcanza la memoria? (2 pts)
**Para qué sirve:** decide si la campaña cabe completa o hay que ir a descargar a la mitad.

Un intervalo cómodo es **diez lecturas por ciclo** (Δt = P/10). Con ese intervalo, ¿cuántos **días** tarda en llenarse
la memoria de `MEMORIA` lecturas? (un día tiene 86 400 s)

In [ ]:
# ✎ 4a · Tu resultado: el número que calculaste (punto decimal, al menos 3 cifras significativas)
dias_memoria = None   # ✎ escribe aquí
# ✎ 4b · La sustitución con TUS números, el resultado y la unidad
SUST_4 = ""   # ✎ escribe aquí
# ✎ 4c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_4 = ""   # ✎ escribe aquí

revisa(4, dias_memoria, SUST_4, PRED_4, datos={'tu dato MEMORIA': MEMORIA, 'tu dato P_S / 10': P_S / 10}, unidades=['día'])

### Cierre · El intervalo de tu campaña (1 pt)
Elige el intervalo, en segundos, que programarás en el RC-5 para registrar tu escenario **7 días**. Debe cumplir tres
reglas: no menor que 10 s (lo más rápido del RC-5), no mayor que tu `dt_max_s`, y que la memoria alcance los 7 días.
Después justifica tu elección con tus números (al menos 20 palabras).

In [ ]:
INTERVALO_CAMPANA_S = None   # ✎ el intervalo en segundos
JUSTIFICACION = ""   # ✎ escribe aquí

revisa_intervalo(INTERVALO_CAMPANA_S, P_S)
revisa_texto('Justificación', JUSTIFICACION, {'el intervalo que elegiste': INTERVALO_CAMPANA_S,
             'tu intervalo máximo por Nyquist': dt_max_s})

## Parte B · Práctica (80 min)
**Objetivo:** ver el aliasing en datos reales y comprobar que la fórmula del ejercicio 2 lo predice.

La temperatura de Guadalajara tiene un ciclo de **24 h** (día y noche). Vas a registrarla cada `DT_B_H` horas, más de
12 h, así que el registro miente. Los datos son 60 días reales, hora por hora
([Open-Meteo](https://open-meteo.com/), licencia CC BY 4.0); las fechas dependen de tu código.

| Bloque | min | Qué haces |
|---|---|---|
| B1 · Datos | 10 | Descargas los 60 días |
| B2 · Predicción | 15 | Calculas el periodo del ciclo falso que debe aparecer |
| B3 · Registro y conteo | 25 | Registras cada `DT_B_H` h, cuentas los ciclos falsos en la gráfica y calculas el periodo obtenido |
| B4 · Conclusión | 20 | Comparas predicho contra obtenido, con tus números |
| Tabla y captura | 10 | Descargas el CSV y lo importas en tu hoja de resultados |

In [ ]:
import json, urllib.request, datetime as dt
inicio = dt.date(2025, 1, 6) + dt.timedelta(weeks=int(alumno.codigo) % 40)
fin = inicio + dt.timedelta(days=59)
url = ("https://archive-api.open-meteo.com/v1/archive?latitude=20.67&longitude=-103.35"
       f"&start_date={inicio}&end_date={fin}&hourly=temperature_2m&timezone=America%2FMexico_City")
try:
    with urllib.request.urlopen(url, timeout=30) as resp:
        T_hora = np.array(json.load(resp)["hourly"]["temperature_2m"], dtype=float)
    FUENTE = "Open-Meteo (ERA5), Guadalajara"
except Exception as e:
    print("⚠ No se pudo descargar de Open-Meteo:", type(e).__name__, e)
    print("⚠ Se usan datos SINTÉTICOS. Vuelve a ejecutar esta celda con internet antes de entregar.")
    hh = np.arange(60 * 24)
    T_hora = (21 + 7 * np.sin(2 * np.pi * (hh - 9) / 24) + 2 * np.sin(2 * np.pi * hh / (24 * 9))
              + alumno.rng("ruido").normal(0, 0.7, hh.size))
    FUENTE = "SINTÉTICOS (sin conexión)"
HORAS = len(T_hora)
print(f"B1 · Del {inicio} al {fin} · {HORAS} lecturas horarias · fuente: {FUENTE}")
print(f"     Tu intervalo de registro para la práctica: DT_B_H = {DT_B_H} h")

### B2 · Predicción
Con la fórmula del ejercicio 2, ahora en horas: periodo real 24 h, registro cada `DT_B_H` h. ¿De cuántas horas será
el ciclo falso?

In [ ]:
T_aparente_pred_h = None   # ✎ escribe aquí

completa(T_aparente_pred_h, 'la predicción del ciclo falso')
print(f'Predicción: un ciclo falso de {T_aparente_pred_h:.1f} h ({T_aparente_pred_h / 24:.1f} días)')

### B3 · Registro y conteo
La celda registra la temperatura cada `DT_B_H` horas y dibuja las dos series. **Cuenta en la gráfica de abajo cuántos
ciclos completos del registro (de máximo a máximo) caben en los 60 días** y escríbelo en `CICLOS_CONTADOS`.
El periodo obtenido es `HORAS / CICLOS_CONTADOS`.

In [ ]:
inicio_h = int(alumno.codigo) % DT_B_H
t_reg = np.arange(inicio_h, HORAS, DT_B_H)
T_reg = T_hora[t_reg]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
a1.plot(np.arange(HORAS) / 24, T_hora, color="#9DB7D5", lw=0.8)
a1.set_ylabel("T (°C)"); titulo(a1, "Temperatura real, hora por hora", alumno)
a2.plot(t_reg / 24, T_reg, "-o", color="#D9822B", ms=3)
a2.set_ylabel("T (°C)"); a2.set_xlabel("día"); a2.set_xticks(range(0, 61, 5)); a2.grid(alpha=0.3)
a2.set_title(f"Registrada cada {DT_B_H} h: cuenta los ciclos")
firma(fig, alumno); plt.show()

In [ ]:
CICLOS_CONTADOS = None   # ✎ cuántos ciclos completos contaste

completa(CICLOS_CONTADOS, "el número de ciclos que contaste")
T_aparente_obt_h = HORAS / CICLOS_CONTADOS
T_fft_h = periodo_dominante(t_reg, T_reg, 2 * DT_B_H, HORAS / 2)
RESULTADOS = {"dt_max_s": dt_max_s, "T_aparente_s": T_aparente_s, "error_cuant": error_cuant,
              "dias_memoria": dias_memoria, "intervalo_campana_s": INTERVALO_CAMPANA_S, "DT_B_h": DT_B_H,
              "T_aparente_pred_h": T_aparente_pred_h, "T_aparente_obt_h": T_aparente_obt_h}
print(f"Periodo obtenido por conteo: {T_aparente_obt_h:.1f} h · predicho: {T_aparente_pred_h:.1f} h")
print(f"(El cuaderno lo estima también con la transformada de Fourier: {T_fft_h:.1f} h)")

### B4 · Conclusión
¿El ciclo falso que contaste coincide con el que predijiste? ¿Qué le dirías a quien programó el registro cada
`DT_B_H` h? Al menos 25 palabras, con tus dos valores.

In [ ]:
CONCLUSION_B = ""   # ✎ escribe aquí

revisa_texto('Conclusión de la práctica', CONCLUSION_B,
             {'el periodo predicho': T_aparente_pred_h, 'el periodo obtenido': T_aparente_obt_h}, minimo=25)

### Tabla y captura · CSV para tu hoja de resultados
La celda descarga `S2_<código>.csv`. En tu hoja de resultados: abre la pestaña **CSV** → **Archivo → Importar →
Subir** → elige el archivo → **Reemplazar la hoja actual**. La pestaña Resultados se llena sola.

In [ ]:
ruta = f"S2_{alumno.codigo}.csv"
exportar_csv(ruta, alumno, RESULTADOS,
             {"dia": t_reg / 24, "T_registrada_C": T_reg},
             extra={"fuente": FUENTE, "periodo": f"{inicio} a {fin}", "escenario": LUGAR})
print("Archivo listo:", ruta, "· huella", huella(alumno, RESULTADOS))
try:
    from google.colab import files
    files.download(ruta)
except ImportError:
    pass

## Siguiente paso · Las campañas de 7 días
El grupo tiene **tres** registradores RC-5, así que se harán tres campañas. Con las fichas de todos, el profesor elige
tres lugares de la planta y, para cada uno, el intervalo propuesto que mejor cumple las tres reglas.
1. Copia tu ficha de campaña (celda de abajo) en tu hoja de resultados, en la pregunta 2.
2. Quienes coloquen un registrador lo programan en ElitechLog con el intervalo elegido, toman una foto del
   registrador colocado con una tarjeta con su nombre a la vista y no lo mueven en 7 días.
3. Los tres archivos se comparten con todo el grupo: los analizarás en el proyecto integrador con las
   herramientas de ruido, tendencias y pronóstico de las sesiones 5 y 6.

In [ ]:
print(f"Ficha de campaña · {alumno.nombre} · {alumno.codigo} · {LUGAR} · {VARIABLE} · intervalo {INTERVALO_CAMPANA_S} s · {CAMPANA_DIAS} días · {MEMORIA * INTERVALO_CAMPANA_S / 86400:.1f} días de memoria")

## Cierre del cuaderno
Antes de entregar: **Entorno de ejecución → Reiniciar y ejecutar todo**. Todo debe salir ✔.

In [ ]:
for k, ok in REGISTRO.items():
    print(("✔" if ok else "✘"), k if isinstance(k, str) else f"Ejercicio {k}")
print(f"\n{alumno.nombre} · {alumno.codigo} · {SESION} · {ahora()} · fuente: {FUENTE} · huella {huella(alumno, RESULTADOS)}")